In [ ]:
%run ./dataengine.ipynb

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Аугментация данных
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Приводим к размеру MobileNetV2
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Пути к папкам
train_dir = "data/split_ttv_dataset_type_of_plants/Train_Set_Folder"
val_dir = "data/split_ttv_dataset_type_of_plants/Validation_Set_Folder"
test_dir = "data/split_ttv_dataset_type_of_plants/Test_Set_Folder"

# Загружаем данные
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Проверим классы
print("Классы:", train_data.classes)


In [ ]:
import torch.nn as nn
import torchvision.models as models

# Загружаем предобученную MobileNetV2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Используем GPU, если есть
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model = model.to(device)

# Заменяем последний слой классификации
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512), 
    nn.ReLU(), 
    nn.Dropout(0.3), 
    nn.Linear(512, 7)  # 7 классов
)

model = model.to(device)

# Проверяем структуру модели
print(model)


In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Преобразования изображений
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Размер входа для MobileNetV2
    transforms.ToTensor(),           # Перевод в тензор
    transforms.Normalize([0.5], [0.5])  # Нормализация
])

# Пути к данным
train_path = "./data/split_ttv_dataset_type_of_plants/Train_Set_Folder"
val_path = "./data/split_ttv_dataset_type_of_plants/Validation_Set_Folder"

# Загрузка данных
train_dataset = ImageFolder(root=train_path, transform=transform)
val_dataset = ImageFolder(root=val_path, transform=transform)

# Создание DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Проверка классов
print("Классы:", train_dataset.classes)


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Определяем устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель MobileNetV2
model = timm.create_model("mobilenetv2_100", pretrained=True, num_classes=7)
model = model.to(device)

# Функция потерь
criterion = nn.CrossEntropyLoss()

# Оптимизатор (лучше AdamW, но можно и SGD)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)


In [ ]:
num_epochs = 10  # Количество эпох

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.00001

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Обнуление градиентов
        optimizer.zero_grad()

        # Прямой проход
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Обратное распространение ошибки
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Эпоха {epoch+1}/{num_epochs}, Потери: {running_loss / len(train_loader)}")

print("Обучение завершено!")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Преобразования для тестовых изображений (без аугментации)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Приводим к размеру модели
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Загружаем тестовые данные
test_dir = "./data/split_ttv_dataset_type_of_plants/Test_Set_Folder"
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

# Создаём DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Взять 1 батч данных
images, labels = next(iter(test_loader))

# Визуализация нескольких изображений
fig, axes = plt.subplots(1, 5, figsize=(10, 5))
for i in range(5):
    img = images[i].permute(1, 2, 0).numpy()  # Перекладываем каналы
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])  # Де-нормализация
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f"Класс: {test_dataset.classes[labels[i]]}")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

    # Функция для визуализации изображений всех классов
def visualize_all_classes(test_loader, test_dataset, num_classes=5):
        # Перебор всех классов и создание подграфиков для каждого
        fig, axes = plt.subplots(num_classes, 5, figsize=(15, 3 * num_classes))
        
        # Для каждого класса
        for class_idx in range(num_classes):
            # Получаем изображения и метки для данного класса
            class_images, class_labels = [], []
            for images, labels in test_loader:
                for i in range(len(labels)):
                    if labels[i] == class_idx:  # Проверяем, соответствует ли метка классу
                        class_images.append(images[i])
                        class_labels.append(labels[i])

            # Визуализация картинок для данного класса
            for i in range(min(5, len(class_images))):  # Ограничиваем до 5 изображений на класс
                img = class_images[i].permute(1, 2, 0).numpy()  # Перекладываем каналы
                img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])  # Де-нормализация
                img = np.clip(img, 0, 1)

                ax = axes[class_idx, i]
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(f"Класс: {test_dataset.classes[class_idx]}")

        plt.tight_layout()
        plt.show()

    # Пример вызова функции
visualize_all_classes(test_loader, test_dataset, num_classes=len(test_dataset.classes))


In [ ]:
import torch
from sklearn.metrics import accuracy_score

# Переключаем модель в режим оценки
model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)  # Берём индекс класса с наибольшей вероятностью
        
        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(preds.cpu().numpy())

# Оцениваем точность
accuracy = accuracy_score(true_labels, pred_labels)
print(f"Точность на тестовых данных: {accuracy:.4f}")


In [ ]:
import torch
import torch.nn as nn
import timm

# Количество классов (должно соответствовать обученной модели)
num_classes_old = 7  # Укажите правильное количество классов у старой модели
num_classes_new = 7     # Новое количество классов

# Загружаем предобученную MobileNetV2
model = timm.create_model("mobilenetv2_100", pretrained=False, num_classes=num_classes_old)

# Загружаем старые веса
state_dict = torch.load("mobilenetv2_model.pth", map_location="cpu")

# Загружаем веса в модель (strict=False, чтобы избежать ошибок)
model.load_state_dict(state_dict, strict=False)

# Сохраняем фичи из старого классификатора
in_features = model.classifier.in_features

# Создаем новый классификатор
model.classifier = nn.Linear(in_features, num_classes_new)

# (Опционально) Заново обучить `classifier` перед сохранением
# Здесь можно написать код для дообучения последнего слоя

# Сохраняем новую обученную модель
torch.save(model.state_dict(), "mobilenetv2_finetuned.pth")

print("Обученная модель загружена, обновлена и сохранена!")


In [ ]:
# Количество классов (указывается такое же, как при обучении)
num_classes = 7  

# Создаем модель с таким же backbone
model = timm.create_model("mobilenetv2_100", pretrained=False, num_classes=num_classes)

# Загружаем веса
state_dict = torch.load("mobilenetv2_finetuned.pth", map_location="cpu")
model.load_state_dict(state_dict)

# Переводим модель в режим предсказания (выключаем обучение)
model.eval()

print("Модель успешно загружена и готова к предсказаниям!")


In [ ]:
import requests
from io import BytesIO
from PIL import Image
import torch
from torchvision import transforms

# Загружаем модель
model = torch.load("mobilenetv2_finetuned_updated.pth", map_location=torch.device('cpu'))
model.eval()

# Преобразование изображений
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Классы
classes = ["aloevera", "cucumber", "kale", "longbeans", "peper chili", "shallot", "spinach"]

def classify_image(image_url):
    try:
        response = requests.get(image_url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = transform(image).unsqueeze(0)
        
        with torch.no_grad():
            outputs = model(image)
            _, predicted = torch.max(outputs, 1)
        
        print(f"Предсказанный класс: {classes[predicted.item()]}")
    except Exception as e:
        print("Ошибка загрузки или обработки изображения:", str(e))

if __name__ == "__main__":
    while True:
        image_url = input("Введите ссылку на изображение (или 'exit' для выхода): ")
        if image_url.lower() == 'exit':
            break
        classify_image(image_url)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import MobileNet_V2_Weights

# Определяем устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель
model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier = torch.nn.Linear(1280, 7)  # Устанавливаем 7 классов
model.load_state_dict(torch.load("mobilenetv2_finetuned.pth", map_location=device), strict=False)  # Загружаем веса
model.to(device)  # Переносим на нужное устройство

# Замораживаем все слои, кроме classifier
for param in model.features.parameters():
    param.requires_grad = False  

# Оптимизатор и функция потерь
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.00001)
criterion = torch.nn.CrossEntropyLoss()

# Обучение classifier 2–3 эпохи
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total
    print(f"Эпоха {epoch+1}/{num_epochs}, Потери: {running_loss / len(train_loader)}, Точность на train: {train_acc:.4f}")

    # Валидация
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"==> Валидация: Потери {val_loss / len(val_loader)}, Точность {val_acc:.4f}")

    # Автосохранение каждые 2 эпохи
    if (epoch + 1) % 2 == 0:
        torch.save(model.state_dict(), "mobilenetv2_finetuned_checkpoint.pth")
        print("Модель сохранена (checkpoint)")

# Размораживаем все слои и продолжаем обучение всей сети
for param in model.features.parameters():
    param.requires_grad = True  

optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)  # Снижаем lr для всей модели
num_epochs = 7

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total
    print(f"Эпоха {epoch+1}/{num_epochs}, Потери: {running_loss / len(train_loader)}, Точность на train: {train_acc:.4f}")

    # Валидация
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"==> Валидация: Потери {val_loss / len(val_loader)}, Точность {val_acc:.4f}")

    # Автосохранение каждые 2 эпохи
    if (epoch + 1) % 2 == 0:
        torch.save(model.state_dict(), "mobilenetv2_finetuned_checkpoint.pth")
        print("Модель сохранена (checkpoint)")

# Финальное сохранение модели
torch.save(model.state_dict(), "mobilenetv2_finetuned_updated.pth")
print("Обучение завершено, модель сохранена!")


In [2]:
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Восстанавливаем MobileNetV2 с нужной архитектурой
model = models.mobilenet_v2(weights=None)  # Без предобученных весов
model.classifier = nn.Linear(1280, 7)  # Количество классов
model.load_state_dict(torch.load("mobilenetv2_finetuned_updated.pth", map_location=device))
model.to(device)
model.eval()


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [3]:
import requests
from io import BytesIO
from PIL import Image
import torch
from torchvision import transforms

# Загружаем модель
model.load_state_dict(torch.load("mobilenetv2_finetuned_updated.pth"))
model.eval()  # Переключаем в режим инференса

# Преобразование изображений
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Классы
classes = ["aloevera", "cucumber", "kale", "longbeans", "peper chili", "shallot", "spinach"]

def classify_image(image_url):
    try:
        response = requests.get(image_url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = transform(image).unsqueeze(0)
        
        with torch.no_grad():
            outputs = model(image)
            _, predicted = torch.max(outputs, 1)
        
        print(f"Предсказанный класс: {classes[predicted.item()]}")
    except Exception as e:
        print("Ошибка загрузки или обработки изображения:", str(e))

if __name__ == "__main__":
    while True:
        image_url = input("Введите ссылку на изображение (или 'exit' для выхода): ")
        if image_url.lower() == 'exit':
            break
        classify_image(image_url)


Ошибка загрузки или обработки изображения: HTTPSConnectionPool(host='bedford.tennessee.edu', port=443): Max retries exceeded with url: /wp-content/uploads/sites/162/2020/08/spinach-leaves-1024x593.jpg (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002206CBCB0E0>: Failed to resolve 'bedford.tennessee.edu' ([Errno 11001] getaddrinfo failed)"))
Предсказанный класс: spinach
Предсказанный класс: shallot
Предсказанный класс: peper chili
Предсказанный класс: kale
Предсказанный класс: longbeans
Предсказанный класс: longbeans
Предсказанный класс: longbeans
Ошибка загрузки или обработки изображения: 403 Client Error: Forbidden for url: https://healthyrecipesblogs.com/wp-content/uploads/2015/07/spicy-green-beans-featured-2021.jpg
Предсказанный класс: longbeans
Ошибка загрузки или обработки изображения: 403 Client Error: Forbidden for url: https://foragerchef.com/wp-content/uploads/2017/07/Eating-the-whole-spinach_-2.jpg
Ошибка загрузки или обработки изображения